In [4]:
import SimpleITK as sitk
import trimesh
import numpy as np
import os
import shutil
import nibabel as nib
import numpy as np
from scipy.ndimage import binary_fill_holes
from scipy.ndimage import label
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

In [ ]:
# 统计nii.gz数据分布
import nibabel as nib
import numpy as np

def get_nii_data_distribution(file_path):
    """
    统计 .nii.gz 文件数据的分布。
    
    参数:
    - file_path (str): .nii.gz 文件的路径
    
    返回:
    - dict: 一个字典，包含每个唯一值及其出现次数
    """
    # 加载 .nii.gz 文件
    img = nib.load(file_path)

    # 获取图像数据为 numpy 数组
    data = img.get_fdata()

    # 使用 np.unique 统计不同值及其出现次数
    unique_values, counts = np.unique(data, return_counts=True)

    # 创建一个字典来存储统计结果
    distribution = dict(zip(unique_values, counts))
    # 打印统计结果
    for value, count in distribution.items():
        print(f"Value: {value}, Count: {count}")
get_nii_data_distribution(r'submit\18757_combine.nii.gz')

## 1.寻找2和18(上牙槽骨和16牙)

### 1.1保存2和18同时出现的切片并保存

In [ ]:
import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
# 定义归一化函数
def normalize(slice_data):
    min_value = np.min(slice_data)
    max_value = np.max(slice_data)
    # 使用公式归一化到 [0, 1]
    normalized_data = (slice_data - min_value) / (max_value - min_value)
    return normalized_data
# 加载原图
original_img = nib.load('submit/18757.nii.gz')
original_data = original_img.get_fdata()

# 加载牙齿标签
label_img = nib.load('submit/18757_combine.nii.gz')
label_data = label_img.get_fdata()

# 创建标签2和标签29的mask
mask_2 = (label_data == 2).astype(np.uint8)  # 标签2的区域
mask_18 = (label_data == 18).astype(np.uint8)  # 标签29的区域

# 创建保存图片的目录
output_dir = "slices_with_labels_2_and_18"
os.makedirs(output_dir, exist_ok=True)
# 是否是第一张切片
first = True

# 按 Z 轴处理每一层
for z in range(original_data.shape[2]):
    slice_original = original_data[:, :, z]  # 原图对应的切片
    # 检查当前切片是否同时有标签2和标签29
    if np.any(mask_2[:, :, z] == 1) and np.any(mask_18[:, :, z] == 1):
        # 获取图像的形状（高度和宽度）
        height, width = slice_original.shape
        # 创建一个与原图形状相同大小的图形
        plt.figure(figsize=(width / 100, height / 100))  # /100是为了缩放以适应实际显示
        
        # 显示原始切片（灰度图）
        plt.imshow(slice_original.T, cmap='gray')
        
        # 在标签2的区域涂成绿色
        plt.imshow(mask_2[:, :, z].T, cmap='Greens', alpha=0.5)
        
        # 在标签29的区域涂成红色
        plt.imshow(mask_18[:, :, z].T, cmap='Reds', alpha=0.5)
        
        # 保存带标记的切片图像
        slice_filename = os.path.join(output_dir, f"slice_{z:03d}_marked.png")
        plt.axis('off')  # 不显示坐标轴
        plt.savefig(slice_filename, bbox_inches='tight', pad_inches=0)
        plt.close()
        
        print(f"保存切片: {slice_filename}")

###  1.2 寻找到牙槽骨和16牙第一次相接时的切片

In [2]:
import os
import nibabel as nib
import numpy as np
import cv2

# 加载原图
original_img = nib.load('submit/18757.nii.gz')
original_data = original_img.get_fdata()

# 加载牙齿标签
label_img = nib.load('submit/18757_combine.nii.gz')
label_data = label_img.get_fdata()

# 创建标签2和标签29的mask
mask_2 = (label_data == 2).astype(np.uint8)  # 标签2的区域
mask_18 = (label_data == 18).astype(np.uint8)  # 标签29的区域

# 八个方向（上下左右及四个对角线）
directions = [(-1, 0), (1, 0), (0, -1), (0, 1),]  # 四个对角线方向

# 用于记录第一次相接的切片编号
first_connection_z = -1

# 按 Z 轴处理每一层
for z in range(original_data.shape[2]):
    slice_original = original_data[:, :, z]  # 原图对应的切片
    
    # 仅处理同时包含标签2和标签29的切片
    if np.any(mask_2[:, :, z] == 1) and np.any(mask_18[:, :, z] == 1):
        # 提取标签29区域的轮廓
        contours, _ = cv2.findContours(mask_18[:, :, z].T, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # 遍历轮廓中的每个点
        for contour in contours:
            for point in contour:
                # 轮廓点坐标
                x, y = point[0]
                
                # 检查该点的八个方向是否有标签2
                for dx, dy in directions:
                    nx, ny = x + dx, y + dy
                    
                    # 确保索引在有效范围内
                    if 0 <= nx < mask_18.shape[0] and 0 <= ny < mask_18.shape[1]:
                        if mask_2[nx, ny, z] == 1:  # 如果标签2出现在邻域
                            first_connection_z = z  # 记录第一次相接的切片编号
                            break
                if first_connection_z != -1:
                    break
            if first_connection_z != -1:
                break
        if first_connection_z != -1:
            break

# 输出第一次相接的切片编号
if first_connection_z != -1:
    print(f"第一次标签2和标签29相接的切片编号是: {first_connection_z}")
else:
    print("没有找到标签2和标签29相接的切片。")


第一次标签2和标签29相接的切片编号是: 256


### 1.3 可视化轮廓和相交点的过程

In [ ]:
import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import cv2

# 加载原图
original_img = nib.load('submit/18757.nii.gz')
original_data = original_img.get_fdata()

# 加载牙齿标签
label_img = nib.load('submit/18757_combine.nii.gz')
label_data = label_img.get_fdata()

# 创建标签2和标签29的mask
mask_2 = (label_data == 2).astype(np.uint8)  # 标签2的区域
mask_29 = (label_data == 18).astype(np.uint8)  # 标签29的区域

# 创建保存图片的目录
output_dir = "slices_with_labels_2_and_18"
os.makedirs(output_dir, exist_ok=True)

# 八个方向（上下左右及四个对角线）
directions = [(-1, 0), (1, 0), (0, -1), (0, 1),]  # 四个对角线方向

# 按 Z 轴处理每一层
for z in range(original_data.shape[2]):
    slice_original = original_data[:, :, z]  # 原图对应的切片
    
    # 仅处理同时包含标签2和标签29的切片
    if np.any(mask_2[:, :, z] == 1) and np.any(mask_29[:, :, z] == 1):
        slice_mask_29 = mask_29[:, :, z]
        # 翻转切片的坐标顺序 (x, y) -> (y, x)
        slice_mask_29_transposed = slice_mask_29.T  # 这样转换为 (y, x) 顺序
        # 提取标签29区域的轮廓
        contours, _ = cv2.findContours(slice_mask_29_transposed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        
        # 将原始图像转换为 8 位无符号整数类型
        slice_original_uint8 = np.uint8(slice_original)

        # 创建一张空白图像（背景为灰度）
        plt.figure(figsize=(8, 8))
        plt.imshow(slice_original_uint8, cmap='gray')
        
        # 在图像上绘制标签2的区域（绿色）
        plt.imshow(mask_2[:, :, z], cmap='Greens', alpha=0.5)
        plt.imshow(mask_29[:, :, z], cmap='Reds', alpha=0.5)
        
        
        # 遍历轮廓中的每个点并绘制
        for contour in contours:
            for point in contour:
                x, y = point[0]
                
                # 绘制轮廓点（红色圆点）
                plt.plot(y, x, 'ro', markersize=5)
                
                # 检查该点的八个方向是否有标签2
                for dx, dy in directions:
                    nx, ny = x + dx, y + dy
                    
                    # 确保索引在有效范围内
                    if 0 <= nx < mask_29.shape[0] and 0 <= ny < mask_29.shape[1]:
                        if mask_2[nx, ny, z] == 1:  # 如果标签2出现在邻域
                            # 在图像上标记该方向的点（蓝色圆点）
                            plt.plot(ny, nx, 'bo', markersize=3)
                        else:
                            # 在图像上标记该方向的点（绿色圆点）
                            plt.plot(ny, nx, 'go', markersize=3)
        
        # 保存该切片的图像
        slice_filename = os.path.join(output_dir, f"slice_{z:03d}_marked1.png")
        plt.axis('off')  # 不显示坐标轴
        plt.savefig(slice_filename, bbox_inches='tight', pad_inches=0)
        plt.close()
        
        print(f"保存切片: {slice_filename}")

print("处理完成。")


## 2.定位牙齿的最小外接矩形，并得到所需要的层面

### 2.1 定位牙齿的最小外接矩形

In [12]:
def find_intersection_points(left_1, right_1):
    # 计算两点之间的欧几里得距离
    distance = np.linalg.norm(np.array(right_1) - np.array(left_1))

    # 计算步长为1的离散点数
    num_points = int(distance)  # 步长为1，每一小段代表一个点

    # 计算每个点的坐标
    points = []
    for i in range(num_points + 1):
        t = i / num_points  # t从0到1
        x = int(round(left_1[0] + t * (right_1[0] - left_1[0])))  # x坐标
        y = int(round(left_1[1] + t * (right_1[1] - left_1[1])))  # y坐标
        points.append((x, y))
    return points
def find_enter_exit_point(points,target_slice):
    # 初始化进入和离开的点
    enter_point = None
    exit_point = None

    # 遍历所有点
    for i in range(len(points)):
        point = points[i]
        # 如果点在目标切片上，并且进入点还未确定，则将其设置为进入点
        if target_slice[point[0], point[1]] != 0 and enter_point is None:
            enter_point = point
        # 如果点在目标切片上，并且进入点已确定，则将其设置为离开点
        elif target_slice[point[0], point[1]] == 0 and enter_point is not None:
            exit_point = point
            break
    if enter_point is None:
        enter_point = points[0]
    if exit_point is None:
        exit_point = points[-1]
    return enter_point, exit_point
target_slice = mask_18[:, :, first_connection_z]
height, width = target_slice.shape
# 查找轮廓
contours, _ = cv2.findContours(target_slice, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 计算最小外接矩形
rect = cv2.minAreaRect(contours[0])
box = cv2.boxPoints(rect)  # 获取矩形四个顶点坐标
box = np.int0(box)  # 转换为整数
# box的四个顶点是逆时针方向排列，以y x方式排列
# box 是逆时针排列的 (y, x) 格式，所以我们直接使用它进行绘制
# 提取矩形四个点
# box 是逆时针排列的 (y, x) 格式，获取左上、左下、右下、右上的坐标
left_up = (box[0][1], box[0][0])  # (x1, y1)
left_down = (box[1][1], box[1][0])  # (x2, y2)
right_down = (box[2][1], box[2][0])  # (x3, y3)
right_up = (box[3][1], box[3][0])  # (x4, y4)

# 计算均匀分布的点
# 左侧的均匀分布点 left_1, left_2, left_3
left_1 = (left_up[0] + (left_down[0] - left_up[0]) * 1 / 4, left_up[1] + (left_down[1] - left_up[1]) * 1 / 4)
left_2 = (left_up[0] + (left_down[0] - left_up[0]) * 2 / 4, left_up[1] + (left_down[1] - left_up[1]) * 2 / 4)
left_3 = (left_up[0] + (left_down[0] - left_up[0]) * 3 / 4, left_up[1] + (left_down[1] - left_up[1]) * 3 / 4)

# 右侧的均匀分布点 right_1, right_2, right_3
right_1 = (right_up[0] + (right_down[0] - right_up[0]) * 1 / 4, right_up[1] + (right_down[1] - right_up[1]) * 1 / 4)
right_2 = (right_up[0] + (right_down[0] - right_up[0]) * 2 / 4, right_up[1] + (right_down[1] - right_up[1]) * 2 / 4)
right_3 = (right_up[0] + (right_down[0] - right_up[0]) * 3 / 4, right_up[1] + (right_down[1] - right_up[1]) * 3 / 4)

# 创建一个与原图形状相同大小的图形
plt.figure(figsize=(width/100, height/100))  # /100是为了缩放以适应实际显示

# 显示原始切片（灰度图）
plt.imshow(slice_original.T, cmap='gray')

# 在标签29的区域涂成红色
plt.imshow(target_slice.T, cmap='Reds', alpha=0.5)

# 使用 plt.plot 绘制矩形线条，绿色，线宽为2
rectangle_x = [left_up[0], left_down[0], right_down[0], right_up[0], left_up[0]]  # x 坐标
rectangle_y = [left_up[1], left_down[1], right_down[1], right_up[1], left_up[1]]  # y 坐标
# plt.plot(rectangle_x, rectangle_y, color='green', linewidth=2)

# 绘制蓝色连接线：将左侧的均匀分点与右侧对应点连接
# left_1 -> right_1, left_2 -> right_2, left_3 -> right_3
left_points = [left_1, left_2, left_3]
right_points = [right_1, right_2, right_3]

# 提取坐标
left_x = [point[0] for point in left_points]
left_y = [point[1] for point in left_points]
right_x = [point[0] for point in right_points]
right_y = [point[1] for point in right_points]

# 使用 plt.plot 绘制蓝色连接线
# plt.plot([left_x[0], right_x[0]], [left_y[0], right_y[0]], color='blue', linewidth=2)  # left_1 -> right_1
# plt.plot([left_x[1], right_x[1]], [left_y[1], right_y[1]], color='blue', linewidth=2)  # left_2 -> right_2
# plt.plot([left_x[2], right_x[2]], [left_y[2], right_y[2]], color='blue', linewidth=2)  # left_3 -> right_3

# 寻找交点
point_1 = find_intersection_points(left_1, right_1)
point_2 = find_intersection_points(left_2, right_2)
point_3 = find_intersection_points(left_3, right_3)
enter_1, exit_1 = find_enter_exit_point(point_1, target_slice)
enter_2, exit_2 = find_enter_exit_point(point_2, target_slice)
enter_3, exit_3 = find_enter_exit_point(point_3, target_slice)

# 画线段散点图
# plt.scatter([point[0] for point in point_1], [point[1] for point in point_1], color='blue', marker='o',s=1)
# plt.scatter([point[0] for point in point_2], [point[1] for point in point_2], color='blue', marker='o',s=1)
# plt.scatter([point[0] for point in point_3], [point[1] for point in point_3], color='blue', marker='o',s=1)

# 画交点散点图
plt.scatter(enter_1[0], enter_1[1], color='yellow', marker='o',s=1)
plt.scatter(exit_1[0], exit_1[1], color='yellow', marker='o',s=1)
plt.scatter(enter_2[0], enter_2[1], color='yellow', marker='o',s=1)
plt.scatter(exit_2[0], exit_2[1], color='yellow', marker='o',s=1)
plt.scatter(enter_3[0], enter_3[1], color='yellow', marker='o',s=1)
plt.scatter(exit_3[0], exit_3[1], color='yellow', marker='o',s=1)

# 保存带标记的切片图像
output_dir = 'slices_with_labels_2_and_18'
slice_filename = os.path.join(output_dir, f"points_matplotlib.png")
plt.axis('off')  # 不显示坐标轴
plt.savefig(slice_filename, bbox_inches='tight', pad_inches=0)
plt.close()

print(f"保存切片: {slice_filename}")

C:\Users\86188\AppData\Local\Temp\ipykernel_6108\4189943177.py:44: DeprecationWarning: `np.int0` is a deprecated alias for `np.intp`.  (Deprecated NumPy 1.24)
  box = np.int0(box)  # 转换为整数


保存切片: slices_with_labels_2_and_18\points_matplotlib.png


In [28]:
import numpy as np

# 假设已经给定left_1和right_1
left_1 = (10, 20)  # 示例点，实际情况用你的数据替换
right_1 = (50, 80)  # 示例点，实际情况用你的数据替换

# 计算两点之间的欧几里得距离
distance = np.linalg.norm(np.array(right_1) - np.array(left_1))

# 计算步长为1的离散点数
num_points = int(distance)  # 步长为1，每一小段代表一个点

# 计算每个点的坐标
points = []
for i in range(num_points + 1):
    t = i / num_points  # t从0到1
    x = int(round(left_1[0] + t * (right_1[0] - left_1[0])))  # x坐标
    y = int(round(left_1[1] + t * (right_1[1] - left_1[1])))  # y坐标
    points.append((x, y))

# 打印结果
print("线段上的离散点：")
for point in points:
    print(point)


线段上的离散点：
(10, 20)
(11, 21)
(11, 22)
(12, 22)
(12, 23)
(13, 24)
(13, 25)
(14, 26)
(14, 27)
(15, 28)
(16, 28)
(16, 29)
(17, 30)
(17, 31)
(18, 32)
(18, 32)
(19, 33)
(19, 34)
(20, 35)
(21, 36)
(21, 37)
(22, 38)
(22, 38)
(23, 39)
(23, 40)
(24, 41)
(24, 42)
(25, 42)
(26, 43)
(26, 44)
(27, 45)
(27, 46)
(28, 47)
(28, 48)
(29, 48)
(29, 49)
(30, 50)
(31, 51)
(31, 52)
(32, 52)
(32, 53)
(33, 54)
(33, 55)
(34, 56)
(34, 57)
(35, 58)
(36, 58)
(36, 59)
(37, 60)
(37, 61)
(38, 62)
(38, 62)
(39, 63)
(39, 64)
(40, 65)
(41, 66)
(41, 67)
(42, 68)
(42, 68)
(43, 69)
(43, 70)
(44, 71)
(44, 72)
(45, 72)
(46, 73)
(46, 74)
(47, 75)
(47, 76)
(48, 77)
(48, 78)
(49, 78)
(49, 79)
(50, 80)
